# CSE 4101 — Introduction to Machine Learning
## Practical Lab Assignment 11: Mushroom Edibility Classification

**Goal:** Build an end-to-end categorical classification workflow for the Kaggle Mushroom Classification dataset. This notebook covers all 10 tasks in the assignment.

> **Safety note:** This is a dataset-analysis exercise only. Never use this model to decide whether a wild mushroom is safe to eat. Do not eat mushrooms based on a model prediction.

**Reproducibility:** random seed = `42`. The dataset is expected to be the Kaggle `mushrooms.csv` file with target column `class` (values `e` and `p`). The notebook checks actual column names and handles `?` as a missing-value symbol.


## Setup and data loading

Download `mushrooms.csv` from the assignment's Kaggle page and upload it to the notebook session if it is not already in the working directory. In Google Colab, the upload prompt below will appear when needed.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

csv_path = "mushrooms.csv"
if not os.path.exists(csv_path):
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = next((name for name in uploaded if name.lower().endswith(".csv")), None)
    except ImportError:
        csv_path = input("Enter path to mushrooms.csv: ").strip()

if not csv_path or not os.path.exists(csv_path):
    raise FileNotFoundError("Please place/upload the Kaggle mushrooms.csv file and rerun this cell.")

df = pd.read_csv(csv_path)
print("Dataset shape:", df.shape)
print("Actual columns:", df.columns.tolist())
display(df.head())


## Task 1 — Read the data, category counts and missing-value symbols

In [ ]:
print("Data types:")
display(df.dtypes.rename("dtype").to_frame())
print("\\nMissing values recorded as actual NaN:")
display(df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame())
print("\\nQuestion-mark counts by column:")
question_counts = (df.astype("string") == "?").sum().sort_values(ascending=False)
display(question_counts[question_counts > 0].rename("question_mark_count").to_frame())

print("\\nTarget distribution:")
if "class" not in df.columns:
    raise ValueError("Expected target column 'class' was not found. Inspect actual columns and update target_col.")
target_col = "class"
display(df[target_col].value_counts(dropna=False).rename("count").to_frame())
print("\\nUnique category counts:")
display(df.nunique(dropna=False).sort_values(ascending=False).rename("unique_categories").to_frame())


**Observation (verify against your printed output):** The dataset contains categorical mushroom descriptors and a categorical `class` label. In the standard Kaggle file, `?` is used in some feature fields as a missing-value symbol; it must be treated as missing rather than as a real category.

## Task 2 — Exploratory data analysis and visualizations

In [ ]:
# Replace the question-mark symbol before descriptive analysis.
eda_df = df.replace("?", np.nan).copy()
print("Descriptive summary for categorical columns:")
display(eda_df.describe(include="all").T)

print("Target proportions:")
target_props = eda_df[target_col].value_counts(normalize=True, dropna=False)
display(target_props.rename("proportion").to_frame())

# Plot 1: target class counts
plt.figure(figsize=(6, 4))
sns.countplot(data=eda_df, x=target_col, order=eda_df[target_col].value_counts().index)
plt.title("Mushroom class distribution")
plt.xlabel("Class (e = edible, p = poisonous)")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

# Plot 2: outcome proportions by odor (if available)
if "odor" in eda_df.columns:
    odor_order = eda_df["odor"].value_counts().index
    plt.figure(figsize=(10, 5))
    sns.histplot(data=eda_df, x="odor", hue=target_col, multiple="fill",
                 stat="probability", common_norm=False, shrink=0.85)
    plt.title("Class composition by odor category")
    plt.xlabel("Odor")
    plt.ylabel("Within-category proportion")
    plt.tight_layout()
    plt.show()

# Plot 3: outcome proportions by cap color (if available)
cap_col = "cap-color" if "cap-color" in eda_df.columns else ("cap_color" if "cap_color" in eda_df.columns else None)
if cap_col:
    plot_df = eda_df.dropna(subset=[cap_col, target_col])
    proportions = pd.crosstab(plot_df[cap_col], plot_df[target_col], normalize="index")
    proportions.plot(kind="bar", stacked=True, figsize=(10, 5))
    plt.title(f"Class proportions by {cap_col}")
    plt.xlabel(cap_col)
    plt.ylabel("Within-category proportion")
    plt.legend(title="Class")
    plt.tight_layout()
    plt.show()

# Plot 4: another categorical feature with class composition
candidate = next((c for c in ["gill-color", "gill_size", "gill-size", "habitat"] if c in eda_df.columns), None)
if candidate:
    plot_df = eda_df.dropna(subset=[candidate, target_col])
    proportions = pd.crosstab(plot_df[candidate], plot_df[target_col], normalize="index")
    proportions.plot(kind="bar", stacked=True, figsize=(10, 5))
    plt.title(f"Class proportions by {candidate}")
    plt.xlabel(candidate)
    plt.ylabel("Within-category proportion")
    plt.legend(title="Class")
    plt.tight_layout()
    plt.show()

print("Missing values after converting '?' to NaN:")
display(eda_df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame())


**Three plot-based observations to complete after running:**
1. The target-count plot shows whether the edible and poisonous classes are balanced or imbalanced.
2. Compare the class proportions across odor categories; categories with very different proportions may be informative predictors.
3. Compare the cap-colour (and optional additional feature) stacked bars; differences in class proportions indicate association, not causation.

Use the generated plots and counts to make these observations specific to your run.

## Task 3 — Separate target and handle `?` missing values

In [ ]:
# Standardize question marks to NaN and separate y from X.
clean_df = df.replace("?", np.nan).copy()
target_col = "class"
if target_col not in clean_df:
    raise ValueError("Target column 'class' not found.")
y = clean_df[target_col].astype(str)
X = clean_df.drop(columns=[target_col])

print("Target excluded from feature matrix:", target_col not in X.columns)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Missing values per feature (top):")
display(X.isna().sum().sort_values(ascending=False).head(15).to_frame("missing_count"))
print("Target labels:", sorted(y.unique()))


**Decision:** `class` is excluded from the predictors to prevent target leakage. Question marks are converted to `NaN`; missing categorical values will be imputed using the most frequent category within a preprocessing pipeline fitted on training data only.

## Task 4 — Stratified 70/15/15 split and training-only encoding

In [ ]:
# First split off 15% test; then take 15/85 of the remaining 85% = 15% of total for validation.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "fraction": [len(X_train)/len(X), len(X_val)/len(X), len(X_test)/len(X)],
    "edible_fraction": [(y_train == "e").mean(), (y_val == "e").mean(), (y_test == "e").mean()],
    "poisonous_fraction": [(y_train == "p").mean(), (y_val == "p").mean(), (y_test == "p").mean()]
}, index=["Train", "Validation", "Test"])
display(split_summary)

# Compatibility with both newer and older scikit-learn versions.
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_features = X.columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", encoder)
    ]), categorical_features)],
    remainder="drop"
)
# Fit only on training rows; transform validation/test without fitting on them.
X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)
print("Encoded shapes — train, validation, test:", X_train_enc.shape, X_val_enc.shape, X_test_enc.shape)


**Justification:** Stratification preserves the class proportions approximately in each subset. The test set remains untouched during model selection. The encoder and imputer are fitted only on training rows, preventing information from validation/test data leaking into preprocessing.

## Task 5 — Train Decision Tree, Logistic Regression and Random Forest

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
}
baseline_rows = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train_enc, y_train)
    fitted_models[name] = model
    pred_train = model.predict(X_train_enc)
    pred_val = model.predict(X_val_enc)
    baseline_rows.append({
        "Model": name,
        "Settings": str(model.get_params(deep=False)),
        "Train accuracy": accuracy_score(y_train, pred_train),
        "Validation accuracy": accuracy_score(y_val, pred_val),
        "Validation poisonous recall": recall_score(y_val, pred_val, pos_label="p", zero_division=0),
        "Validation poisonous precision": precision_score(y_val, pred_val, pos_label="p", zero_division=0),
        "Validation poisonous F1": f1_score(y_val, pred_val, pos_label="p", zero_division=0)
    })
baseline_results = pd.DataFrame(baseline_rows).sort_values("Validation poisonous recall", ascending=False)
display(baseline_results[["Model", "Train accuracy", "Validation accuracy",
                          "Validation poisonous recall", "Validation poisonous precision",
                          "Validation poisonous F1"]])
print("Libraries: pandas, NumPy, matplotlib, seaborn, scikit-learn.")
print("Leakage control: all models use features transformed by a preprocessor fitted on X_train only.")


## Task 6 — Tune tree depth and minimum leaf size on validation data

In [ ]:
tuning_rows = []
tuned_candidates = {}
for params in ParameterGrid({
    "max_depth": [3, 5, 8, 12, None],
    "min_samples_leaf": [1, 2, 5, 10, 20]
}):
    candidate_model = DecisionTreeClassifier(random_state=RANDOM_STATE, **params)
    candidate_model.fit(X_train_enc, y_train)
    val_pred = candidate_model.predict(X_val_enc)
    row = {
        **params,
        "train_accuracy": accuracy_score(y_train, candidate_model.predict(X_train_enc)),
        "validation_accuracy": accuracy_score(y_val, val_pred),
        "poisonous_recall": recall_score(y_val, val_pred, pos_label="p", zero_division=0),
        "poisonous_precision": precision_score(y_val, val_pred, pos_label="p", zero_division=0),
        "poisonous_f1": f1_score(y_val, val_pred, pos_label="p", zero_division=0)
    }
    tuning_rows.append(row)
    tuned_candidates[(params["max_depth"], params["min_samples_leaf"])] = candidate_model

tuning_results = pd.DataFrame(tuning_rows)
tuning_results = tuning_results.sort_values(
    ["poisonous_recall", "poisonous_f1", "validation_accuracy"],
    ascending=False
).reset_index(drop=True)
display(tuning_results)
best_row = tuning_results.iloc[0]
best_params = {"max_depth": best_row["max_depth"], "min_samples_leaf": int(best_row["min_samples_leaf"])}
best_tree = DecisionTreeClassifier(random_state=RANDOM_STATE, **best_params)
best_tree.fit(X_train_enc, y_train)
print("Selected tree parameters (validation poisonous recall, then F1, then accuracy):", best_params)


**Interpretation:** The table compares tree complexity settings on validation data. Larger depth and smaller leaf size can fit more complex patterns but may overfit; shallower trees or larger leaves constrain complexity. The chosen configuration prioritizes recall for the poisonous (`p`) class, then F1, then validation accuracy. This is an educational selection rule, not a food-safety rule.

## Task 7 — Compare train and validation performance / complexity

In [ ]:
tuning_plot = tuning_results.copy()
tuning_plot["depth_label"] = tuning_plot["max_depth"].apply(lambda v: "None" if pd.isna(v) else str(v))
plt.figure(figsize=(10, 5))
sns.lineplot(data=tuning_plot, x="max_depth", y="train_accuracy", marker="o", label="Train accuracy")
sns.lineplot(data=tuning_plot, x="max_depth", y="validation_accuracy", marker="o", label="Validation accuracy")
plt.title("Decision Tree train vs validation accuracy by max_depth")
plt.xlabel("Maximum depth (None = unlimited)")
plt.ylabel("Accuracy")
plt.tight_layout()
plt.show()

display(tuning_results[["max_depth", "min_samples_leaf", "train_accuracy",
                        "validation_accuracy", "poisonous_recall", "poisonous_f1"]].head(10))


**Interpretation:** A large gap between training and validation accuracy suggests overfitting. Similar scores suggest better generalization on this split, although validation performance is only an estimate and may vary with the random split.

## Task 8 — Final test evaluation, poisonous recall and confusion matrix

In [ ]:
# Choose among baseline models and tuned tree using validation poisonous recall, then F1, then accuracy.
candidate_models = {**fitted_models, "Tuned Decision Tree": best_tree}
selection_rows = []
for name, model in candidate_models.items():
    p = model.predict(X_val_enc)
    selection_rows.append({
        "Model": name,
        "Validation accuracy": accuracy_score(y_val, p),
        "Poisonous precision": precision_score(y_val, p, pos_label="p", zero_division=0),
        "Poisonous recall": recall_score(y_val, p, pos_label="p", zero_division=0),
        "Poisonous F1": f1_score(y_val, p, pos_label="p", zero_division=0)
    })
selection_table = pd.DataFrame(selection_rows).sort_values(
    ["Poisonous recall", "Poisonous F1", "Validation accuracy"], ascending=False
)
display(selection_table)
selected_name = selection_table.iloc[0]["Model"]
selected_model = candidate_models[selected_name]
print("Selected model:", selected_name)

# The test set is used once for final reporting, after selection.
y_test_pred = selected_model.predict(X_test_enc)
print(classification_report(y_test, y_test_pred, labels=["e", "p"], target_names=["edible (e)", "poisonous (p)"], zero_division=0))
print("Poisonous-class recall:", recall_score(y_test, y_test_pred, pos_label="p", zero_division=0))
cm = confusion_matrix(y_test, y_test_pred, labels=["e", "p"])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["edible (e)", "poisonous (p)"])
disp.plot(values_format="d")
plt.title(f"Test confusion matrix — {selected_name}")
plt.tight_layout()
plt.show()


**Interpretation and limitation:** Poisonous-class recall is the proportion of poisonous test records correctly identified as poisonous. A false negative for the poisonous class is especially concerning in a real-world setting, but even high recall or accuracy does not make the model safe for mushroom identification. Results depend on the dataset and split.

## Task 9 — Predict five held-out records and compare with actual labels

In [ ]:
# Five records from the held-out test set only; their labels are compared after prediction.
n_show = min(5, len(X_test))
sample_X = X_test.iloc[:n_show]
sample_y = y_test.iloc[:n_show]
sample_enc = preprocessor.transform(sample_X)
sample_pred = selected_model.predict(sample_enc)
sample_results = pd.DataFrame({
    "test_row_index": sample_X.index,
    "actual_class": sample_y.values,
    "predicted_class": sample_pred,
    "correct": sample_y.values == sample_pred
})
display(sample_results)

plot_counts = pd.DataFrame({
    "Actual": sample_y.value_counts(),
    "Predicted": pd.Series(sample_pred).value_counts()
}).fillna(0)
plot_counts.plot(kind="bar", figsize=(7, 4))
plt.title("Actual vs predicted classes for five held-out records")
plt.xlabel("Class")
plt.ylabel("Number of records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("Correct predictions out of displayed records:", int(sample_results["correct"].sum()), "of", n_show)


**Interpretation:** The table shows actual and predicted labels for five held-out records. Five examples are illustrative only and are too few to establish general performance; use the full test metrics for evaluation.

## Task 10 — Dataset bias, limitations and conclusion

### Dataset bias and limitations
- **Dataset coverage:** The dataset contains recorded categorical attributes and may not represent all mushroom species, regions, seasons, or field conditions.
- **Data quality / missingness:** `?` values indicate missing or unavailable feature information. Imputation can affect predictions and may hide patterns in missingness.
- **Distribution shift:** Real-world photographs, damaged specimens, unusual growth stages, and environmental variation may differ from the dataset.
- **Feature and label limitations:** Recorded labels and attributes may contain errors or reflect a limited collection process.

### Safety statement
**This model must never guide actual foraging decisions or be used to determine whether a wild mushroom is safe to eat.** Do not consume a mushroom based on this classifier, its predicted label, or its confidence score.

### Conclusion
This notebook follows a reproducible workflow: data profiling, exploratory visualization, missing-value handling, stratified train/validation/test splitting, training-only preprocessing, baseline model comparison, validation-based tree tuning, and final evaluation on a held-out test set. Record the actual metric values printed by your run and discuss any unexpected result using those outputs. Two important real-world limitations are possible dataset bias and poor generalization to mushrooms outside the dataset.

## Submission checklist
- [ ] Run every cell from top to bottom in Google Colab/Jupyter.
- [ ] Ensure the correct Kaggle `mushrooms.csv` file was loaded and confirm the displayed columns.
- [ ] Keep the generated tables and plots in the notebook.
- [ ] Update the three EDA observations and final conclusion with actual numbers from your run.
- [ ] Submit the completed `.ipynb` file (and export to PDF if your instructor requires it).
- [ ] Never use predictions for real-world mushroom safety decisions.
